Auto Annotation of Missing Data

In [1]:
import pandas as pd

In [2]:
file_path = "data/15K_Records_5thJan.csv" # dataset created by combining three separate files

In [ ]:
df = pd.read_csv(file_path)

# Display the first few rows to understand the structure and check annotation patterns
df.head(), df.info()

In [ ]:
# Check distribution of values in the annotated columns
feature_distributions = {
    "fake_news": df["fake_news"].value_counts(dropna=False),
    "hate_speech": df["hate_speech"].value_counts(dropna=False),
    "toxicity": df["toxicity"].value_counts(dropna=False),
}

feature_distributions

In [ ]:
import nltk
nltk.download('stopwords')

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from nltk.corpus import stopwords

# Get the list of German stop words
german_stop_words = stopwords.words('german')


In [7]:
# Function to fill missing annotations for a feature using logistic regression
def fill_missing_annotations(df, feature):
    # Separate data into labeled and unlabeled
    labeled_data = df.dropna(subset=[feature])
    unlabeled_data = df[df[feature].isna()]
    
    # Create the TfidfVectorizer with German stop words
    vectorizer = TfidfVectorizer(stop_words=german_stop_words)
    X_labeled = vectorizer.fit_transform(labeled_data["text"])
    y_labeled = labeled_data[feature]

    # Train a logistic regression classifier
    model = LogisticRegression(max_iter=200)
    model.fit(X_labeled, y_labeled)

    # Vectorize the unlabeled data and predict
    X_unlabeled = vectorizer.transform(unlabeled_data["text"])
    predictions = model.predict(X_unlabeled)

    # Fill the missing values with predictions
    df.loc[df[feature].isna(), feature] = predictions

    # Return the updated dataframe
    return df

In [8]:
# Remove rows where the text column is NaN
df = df.dropna(subset=["text"])

In [ ]:
# Fill missing values for each feature
for feature in ["fake_news", "hate_speech", "toxicity"]:
    df = fill_missing_annotations(df, feature)

In [ ]:
# Verify if all missing values are filled
missing_after_filling = df.isna().sum()

In [ ]:
# Save the completed dataframe to CSV
output_file_path = "data/augmented_annotations.csv"
df.to_csv(output_file_path, index=False)

In [ ]:
missing_after_filling, output_file_path